# Stage 7 - YawDD+ Dash Mouth-Crop CNN Baselines (Google Colab)

This notebook is for **Stage 7 only**.

It assumes Stages 1-6 are already complete and that the following artifacts already exist in Google Drive under:

`/content/drive/MyDrive/Drowsiness_Detection_Colab/`

Required uploaded inputs:

- `dataset/YawDD_plus_reconstructed/Dash/mouth_crops/`
- `artifacts/mappings/yawdd_dash_all_mouth_crops_trainable.csv`
- `artifacts/splits/yawdd_dash_subject_split.csv`

This notebook will:

1. mount Google Drive
2. clone the repository code into `/content/`
3. copy the Stage 7 mouth-crop data from Drive to local Colab storage
4. rewrite the uploaded manifests so they point to local `/content/` paths
5. train the three Stage 7 baselines
6. save results, figures, checkpoints, and summary files back to Google Drive

The labels used here are:

- `no_yawn`
- `yawn`

## Notebook Notes

- This notebook is designed for **Google Colab hosted runtime**.
- For best speed, use **GPU runtime** if available.
- The repository clone URL is configurable near the top of the notebook.
- The default clone URL uses **HTTPS** because that is usually easier in Colab.
- If your Colab runtime is already configured with SSH keys, you can switch `REPO_URL` to the SSH URL.

In [3]:
# 1. Runtime and repository configuration
from pathlib import Path

REPO_URL = "https://github.com/JohnCoffey-commits/Drowsiness-Detection.git"
# If your Colab runtime already has SSH configured, you can switch to:
# REPO_URL = "git@github.com:JohnCoffey-commits/Drowsiness-Detection.git"

REPO_BRANCH = "main"
REPO_DIR = Path("/content/Drowsiness-Detection")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Drowsiness_Detection_Colab")
LOCAL_STAGE7_ROOT = Path("/content/Drowsiness_Detection_Colab_local")
LOCAL_PROJECT_DATA_ROOT = LOCAL_STAGE7_ROOT
LOCAL_MOUTH_CROPS_DIR = LOCAL_PROJECT_DATA_ROOT / "data" / "mouth_crops"
LOCAL_TRAINABLE_MANIFEST = LOCAL_PROJECT_DATA_ROOT / "manifests" / "yawdd_dash_all_mouth_crops_trainable.csv"
LOCAL_SPLIT_MANIFEST = LOCAL_PROJECT_DATA_ROOT / "manifests" / "yawdd_dash_subject_split.csv"

DRIVE_MOUTH_CROPS_DIR = DRIVE_PROJECT_ROOT / "data" / "mouth_crops"
DRIVE_TRAINABLE_MANIFEST = DRIVE_PROJECT_ROOT / "manifests" / "yawdd_dash_all_mouth_crops_trainable.csv"
DRIVE_SPLIT_MANIFEST = DRIVE_PROJECT_ROOT / "manifests" / "yawdd_dash_subject_split.csv"

LOCAL_OUTPUT_ROOT = Path("/content/stage7_outputs")
LOCAL_RESULTS_DIR = LOCAL_OUTPUT_ROOT / "artifacts" / "results"
LOCAL_FIGURES_DIR = LOCAL_OUTPUT_ROOT / "artifacts" / "figures"
LOCAL_CHECKPOINT_DIR = LOCAL_OUTPUT_ROOT / "checkpoints"
LOCAL_REPORTS_DIR = LOCAL_OUTPUT_ROOT / "reports"

DRIVE_RESULTS_DIR = DRIVE_PROJECT_ROOT / "outputs" / "results"
DRIVE_FIGURES_DIR = DRIVE_PROJECT_ROOT / "outputs" / "figures"
DRIVE_CHECKPOINT_DIR = DRIVE_PROJECT_ROOT / "outputs" / "checkpoints"
DRIVE_REPORTS_DIR = DRIVE_PROJECT_ROOT / "outputs" / "reports"

FORCE_RECLONE_REPO = False
FORCE_RECOPY_DATA = False
SEED = 42
DEFAULT_IMAGE_SIZE = 224
DEFAULT_BATCH_SIZE = 32
DEFAULT_EPOCHS = 8
DEFAULT_FREEZE_EPOCHS = 1
DEFAULT_PATIENCE = 2
DEFAULT_LR = 1e-4
NUM_WORKERS = 2


In [4]:
# 2. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project root: /content/drive/MyDrive/Drowsiness_Detection_Colab


In [8]:
# 3. Clone and prepare the repository code
import os
import shutil
import subprocess
import sys

if FORCE_RECLONE_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}. Reusing it.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Working directory: {Path.cwd()}")

Working directory: /content/Drowsiness-Detection


In [9]:
# 4. Install dependencies needed for Stage 7
import importlib
import sys
from pathlib import Path
import subprocess

requirements_path = REPO_DIR / "requirements.txt"
extra_requirements = []
if requirements_path.is_file():
    for raw_line in requirements_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        package_name = line.split("==")[0].split(">=")[0].split("<=")[0].strip()
        if package_name in {"torch", "torchvision"}:
            continue
        extra_requirements.append(line)

if extra_requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra_requirements], check=True)

# Colab usually ships torch/torchvision already. If not, install them here.
missing_core = []
for module_name in ["torch", "torchvision"]:
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_core.append(module_name)

if missing_core:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_core], check=True)

print("Dependency setup finished.")

Dependency setup finished.


In [10]:
# 5. Runtime information
import platform
import torch
import torchvision

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Training will run on CPU.")

print("\nResolved Drive paths:")
print("DRIVE_PROJECT_ROOT:", DRIVE_PROJECT_ROOT)
print("DRIVE_MOUTH_CROPS_DIR:", DRIVE_MOUTH_CROPS_DIR)
print("DRIVE_TRAINABLE_MANIFEST:", DRIVE_TRAINABLE_MANIFEST)
print("DRIVE_SPLIT_MANIFEST:", DRIVE_SPLIT_MANIFEST)
print("DRIVE_RESULTS_DIR:", DRIVE_RESULTS_DIR)
print("DRIVE_FIGURES_DIR:", DRIVE_FIGURES_DIR)
print("DRIVE_CHECKPOINT_DIR:", DRIVE_CHECKPOINT_DIR)
print("DRIVE_REPORTS_DIR:", DRIVE_REPORTS_DIR)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB

Resolved Drive paths:
DRIVE_PROJECT_ROOT: /content/drive/MyDrive/Drowsiness_Detection_Colab
DRIVE_MOUTH_CROPS_DIR: /content/drive/MyDrive/Drowsiness_Detection_Colab/data/mouth_crops
DRIVE_TRAINABLE_MANIFEST: /content/drive/MyDrive/Drowsiness_Detection_Colab/manifests/yawdd_dash_all_mouth_crops_trainable.csv
DRIVE_SPLIT_MANIFEST: /content/drive/MyDrive/Drowsiness_Detection_Colab/manifests/yawdd_dash_subject_split.csv
DRIVE_RESULTS_DIR: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/results
DRIVE_FIGURES_DIR: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/figures
DRIVE_CHECKPOINT_DIR: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/checkpoints
DRIVE_REPORTS_DIR: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/reports


## Drive Inputs

This notebook uses the existing Stage 6 outputs already uploaded to Drive.

The notebook does **not** rebuild the dataset, mouth crops, or subject split.
It only verifies the uploaded Stage 7 inputs and then trains from a **local copy under `/content/`** for speed.

In [11]:
# 6. Resolve Drive paths and verify required inputs exist
required_paths = {
    "DRIVE_PROJECT_ROOT": DRIVE_PROJECT_ROOT,
    "DRIVE_MOUTH_CROPS_DIR": DRIVE_MOUTH_CROPS_DIR,
    "DRIVE_TRAINABLE_MANIFEST": DRIVE_TRAINABLE_MANIFEST,
    "DRIVE_SPLIT_MANIFEST": DRIVE_SPLIT_MANIFEST,
}

print("Checking required Drive inputs...")
missing_required = []
for name, path in required_paths.items():
    exists = path.exists()
    kind = "dir" if path.is_dir() else "file" if path.is_file() else "missing"
    print(f"- {name}: {path} -> {'OK' if exists else 'MISSING'} ({kind})")
    if not exists:
        missing_required.append(f"{name}: {path}")

if missing_required:
    raise FileNotFoundError(
        "Stage 7 notebook cannot start because required Drive inputs are missing.\n"
        "Please upload the required folder/files under /content/drive/MyDrive/Drowsiness_Detection_Colab/ and rerun.\n\n"
        + "\n".join(missing_required)
    )


Checking required Drive inputs...
- DRIVE_PROJECT_ROOT: /content/drive/MyDrive/Drowsiness_Detection_Colab -> OK (dir)
- DRIVE_MOUTH_CROPS_DIR: /content/drive/MyDrive/Drowsiness_Detection_Colab/data/mouth_crops -> OK (dir)
- DRIVE_TRAINABLE_MANIFEST: /content/drive/MyDrive/Drowsiness_Detection_Colab/manifests/yawdd_dash_all_mouth_crops_trainable.csv -> OK (file)
- DRIVE_SPLIT_MANIFEST: /content/drive/MyDrive/Drowsiness_Detection_Colab/manifests/yawdd_dash_subject_split.csv -> OK (file)


In [12]:
# 7. Copy Stage 7 data from Drive to local /content storage
import shutil
from collections import Counter

LOCAL_PROJECT_DATA_ROOT.mkdir(parents=True, exist_ok=True)
(LOCAL_PROJECT_DATA_ROOT / "data").mkdir(parents=True, exist_ok=True)
(LOCAL_PROJECT_DATA_ROOT / "manifests").mkdir(parents=True, exist_ok=True)

expected_drive_count = sum(1 for _ in DRIVE_MOUTH_CROPS_DIR.rglob("*.jpg"))
local_count = sum(1 for _ in LOCAL_MOUTH_CROPS_DIR.rglob("*.jpg")) if LOCAL_MOUTH_CROPS_DIR.exists() else 0

print(f"Drive mouth-crop jpg count: {expected_drive_count}")
print(f"Local mouth-crop jpg count before copy: {local_count}")

if FORCE_RECOPY_DATA and LOCAL_MOUTH_CROPS_DIR.exists():
    shutil.rmtree(LOCAL_MOUTH_CROPS_DIR)
    local_count = 0

if not LOCAL_MOUTH_CROPS_DIR.exists() or local_count != expected_drive_count:
    print("Copying mouth crops from Drive to local Colab storage...")
    if LOCAL_MOUTH_CROPS_DIR.exists():
        shutil.rmtree(LOCAL_MOUTH_CROPS_DIR)
    shutil.copytree(DRIVE_MOUTH_CROPS_DIR, LOCAL_MOUTH_CROPS_DIR)
else:
    print("Local mouth-crop copy already matches Drive count. Reusing local copy.")

print(f"Local mouth-crop jpg count after copy: {sum(1 for _ in LOCAL_MOUTH_CROPS_DIR.rglob('*.jpg'))}")


Drive mouth-crop jpg count: 64202
Local mouth-crop jpg count before copy: 0
Copying mouth crops from Drive to local Colab storage...
Local mouth-crop jpg count after copy: 64202


In [13]:
# 8. Copy manifests locally and rewrite their paths for /content/
import csv
from pathlib import Path

LOCAL_TRAINABLE_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
LOCAL_SPLIT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def remap_to_local(path_str: str) -> str:
    if not path_str:
        return path_str

    path_obj = Path(str(path_str))
    path_text = str(path_obj)

    if path_text.startswith(str(LOCAL_PROJECT_DATA_ROOT)):
        return path_text

    if path_text.startswith(str(DRIVE_PROJECT_ROOT)):
        relative = path_obj.relative_to(DRIVE_PROJECT_ROOT)
        return str(LOCAL_PROJECT_DATA_ROOT / relative)

    mouth_marker = "mouth_crops/"
    if mouth_marker in path_text:
        suffix = path_text.split(mouth_marker, 1)[1]
        return str(LOCAL_MOUTH_CROPS_DIR / suffix)

    filename = path_obj.name
    if filename == "yawdd_dash_all_mouth_crops_trainable.csv":
        return str(LOCAL_TRAINABLE_MANIFEST)
    if filename == "yawdd_dash_subject_split.csv":
        return str(LOCAL_SPLIT_MANIFEST)

    markers = ["data/", "manifests/", "outputs/", "dataset/", "artifacts/", "reports/"]
    for marker in markers:
        idx = path_text.find(marker)
        if idx != -1:
            suffix = path_text[idx:]
            return str(LOCAL_PROJECT_DATA_ROOT / suffix)

    return path_text


def rewrite_manifest(source_path: Path, dest_path: Path) -> int:
    with source_path.open("r", newline="") as fin:
        reader = csv.DictReader(fin)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    for row in rows:
        for key in [
            "image_path",
            "mouth_crop_path",
            "processed_path",
            "original_path",
            "raw_video_path",
            "annotation_txt_path",
        ]:
            if key in row:
                row[key] = remap_to_local(row[key])

    with dest_path.open("w", newline="") as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return len(rows)

trainable_rows = rewrite_manifest(DRIVE_TRAINABLE_MANIFEST, LOCAL_TRAINABLE_MANIFEST)
split_rows = rewrite_manifest(DRIVE_SPLIT_MANIFEST, LOCAL_SPLIT_MANIFEST)
print(f"Rewrote local trainable manifest with {trainable_rows} rows -> {LOCAL_TRAINABLE_MANIFEST}")
print(f"Rewrote local split manifest with {split_rows} rows -> {LOCAL_SPLIT_MANIFEST}")


Rewrote local trainable manifest with 64202 rows -> /content/Drowsiness_Detection_Colab_local/manifests/yawdd_dash_all_mouth_crops_trainable.csv
Rewrote local split manifest with 64202 rows -> /content/Drowsiness_Detection_Colab_local/manifests/yawdd_dash_subject_split.csv


In [14]:
from pathlib import Path

print("LOCAL_PROJECT_DATA_ROOT =", LOCAL_PROJECT_DATA_ROOT)
print("LOCAL_MOUTH_CROPS_DIR =", LOCAL_MOUTH_CROPS_DIR)
print("LOCAL_TRAINABLE_MANIFEST =", LOCAL_TRAINABLE_MANIFEST)
print("LOCAL_SPLIT_MANIFEST =", LOCAL_SPLIT_MANIFEST)

print("\nExists?")
print("LOCAL_MOUTH_CROPS_DIR exists:", LOCAL_MOUTH_CROPS_DIR.exists())
print("LOCAL_TRAINABLE_MANIFEST exists:", LOCAL_TRAINABLE_MANIFEST.exists())
print("LOCAL_SPLIT_MANIFEST exists:", LOCAL_SPLIT_MANIFEST.exists())

if LOCAL_MOUTH_CROPS_DIR.exists():
    subject_dirs = sorted([p.name for p in LOCAL_MOUTH_CROPS_DIR.iterdir() if p.is_dir()])[:5]
    print("\nFirst subject folders under LOCAL_MOUTH_CROPS_DIR:", subject_dirs)

    jpgs = list(LOCAL_MOUTH_CROPS_DIR.rglob("*.jpg"))[:5]
    print("\nFirst jpg paths actually on disk:")
    for p in jpgs:
        print(p)

LOCAL_PROJECT_DATA_ROOT = /content/Drowsiness_Detection_Colab_local
LOCAL_MOUTH_CROPS_DIR = /content/Drowsiness_Detection_Colab_local/data/mouth_crops
LOCAL_TRAINABLE_MANIFEST = /content/Drowsiness_Detection_Colab_local/manifests/yawdd_dash_all_mouth_crops_trainable.csv
LOCAL_SPLIT_MANIFEST = /content/Drowsiness_Detection_Colab_local/manifests/yawdd_dash_subject_split.csv

Exists?
LOCAL_MOUTH_CROPS_DIR exists: True
LOCAL_TRAINABLE_MANIFEST exists: True
LOCAL_SPLIT_MANIFEST exists: True

First subject folders under LOCAL_MOUTH_CROPS_DIR: ['1-FemaleNoGlasses', '1-MaleGlasses', '10-FemaleNoGlasses', '10-MaleGlasses', '11-FemaleGlasses']

First jpg paths actually on disk:
/content/Drowsiness_Detection_Colab_local/data/mouth_crops/10-MaleGlasses/00001367.jpg
/content/Drowsiness_Detection_Colab_local/data/mouth_crops/10-MaleGlasses/00000384.jpg
/content/Drowsiness_Detection_Colab_local/data/mouth_crops/10-MaleGlasses/00000359.jpg
/content/Drowsiness_Detection_Colab_local/data/mouth_crops/10-

In [15]:
# 9. Sanity checks before training
import csv
from collections import Counter, defaultdict

with LOCAL_TRAINABLE_MANIFEST.open("r", newline="") as f:
    trainable_manifest_rows = list(csv.DictReader(f))

with LOCAL_SPLIT_MANIFEST.open("r", newline="") as f:
    split_manifest_rows = list(csv.DictReader(f))

label_set = {row["label"] for row in split_manifest_rows}
print("Labels in split manifest:", label_set)
assert label_set == {"no_yawn", "yawn"}, f"Unexpected labels: {label_set}"

failed_rows = [row for row in trainable_manifest_rows if row.get("crop_method") == "failed"]
assert len(failed_rows) == 0, f"Trainable manifest still contains failed rows: {len(failed_rows)}"

missing_processed = [row["processed_path"] for row in split_manifest_rows if not Path(row["processed_path"]).is_file()]
assert not missing_processed, f"Missing processed_path files: first few -> {missing_processed[:5]}"

split_subjects = defaultdict(set)
split_counts = Counter()
split_label_counts = defaultdict(Counter)
for row in split_manifest_rows:
    split_name = row["split"]
    split_counts[split_name] += 1
    split_subjects[split_name].add(row["subject_id"])
    split_label_counts[split_name][row["label"]] += 1

print("Image counts by split:", dict(split_counts))
print("Subject counts by split:", {k: len(v) for k, v in split_subjects.items()})
print("Label counts by split:")
for split_name in ["train", "val", "test"]:
    print(split_name, dict(split_label_counts[split_name]))

Labels in split manifest: {'yawn', 'no_yawn'}
Image counts by split: {'train': 44156, 'val': 8892, 'test': 11154}
Subject counts by split: {'train': 20, 'val': 4, 'test': 5}
Label counts by split:
train {'no_yawn': 39345, 'yawn': 4811}
val {'no_yawn': 7902, 'yawn': 990}
test {'no_yawn': 9924, 'yawn': 1230}


## Stage 7 Training

The next cells train exactly these three baselines on the **existing Stage 6 split**:

- CNN-1: ResNet18
- CNN-2: MobileNetV2
- CNN-3: EfficientNet-B0

Training setup:

- PyTorch
- transfer learning with pretrained weights if available
- Adam
- learning rate `1e-4`
- batch size `32`, with fallback to `16` on OOM
- weighted cross entropy
- `ReduceLROnPlateau`
- early stopping patience `3`
- freeze backbone first, then fine-tune
- train augmentation only: small rotation, mild brightness/contrast jitter, slight affine scaling
- validation and test: deterministic preprocessing only

In [23]:
# 10. Training utilities
import json
import math
import os
import random
import time
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

LABEL_TO_INDEX = {"no_yawn": 0, "yawn": 1}
INDEX_TO_LABEL = {v: k for k, v in LABEL_TO_INDEX.items()}
MODEL_ROWS = [
    ("CNN-1 (ResNet18)", "resnet18"),
    ("CNN-2 (MobileNetV2)", "mobilenet_v2"),
    ("CNN-3 (EfficientNet-B0)", "efficientnet_b0"),
]


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = select_device()
print("Training device:", device)


class MouthCropDataset(Dataset):
    def __init__(self, rows, transform=None):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        image = Image.open(row["processed_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = LABEL_TO_INDEX[row["label"]]
        return image, torch.tensor(label, dtype=torch.long)


with LOCAL_SPLIT_MANIFEST.open("r", newline="") as f:
    SPLIT_ROWS = list(csv.DictReader(f))

ROWS_BY_SPLIT = {"train": [], "val": [], "test": []}
for row in SPLIT_ROWS:
    ROWS_BY_SPLIT[row["split"]].append(row)


def build_transforms(image_size: int):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    train_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomRotation(8),
        transforms.RandomAffine(degrees=0, scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        normalize,
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tfms, eval_tfms


def make_loaders(batch_size: int, image_size: int, num_workers: int = 2):
    train_tfms, eval_tfms = build_transforms(image_size)
    datasets = {
        "train": MouthCropDataset(ROWS_BY_SPLIT["train"], transform=train_tfms),
        "train_eval": MouthCropDataset(ROWS_BY_SPLIT["train"], transform=eval_tfms),
        "val": MouthCropDataset(ROWS_BY_SPLIT["val"], transform=eval_tfms),
        "test": MouthCropDataset(ROWS_BY_SPLIT["test"], transform=eval_tfms),
    }
    loaders = {}
    for split_name, dataset in datasets.items():
        loaders[split_name] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split_name == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders


def build_model(model_name: str, pretrained: bool = True):
    pretrained_used = pretrained
    try:
        if model_name == "resnet18":
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            model = models.resnet18(weights=weights)
            model.fc = nn.Linear(model.fc.in_features, 2)
        elif model_name == "mobilenet_v2":
            weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
            model = models.mobilenet_v2(weights=weights)
            model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
        elif model_name == "efficientnet_b0":
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            model = models.efficientnet_b0(weights=weights)
            model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
        else:
            raise ValueError(f"Unsupported model: {model_name}")
    except Exception as exc:
        print(f"Pretrained weights unavailable for {model_name}: {exc}. Falling back to random init.")
        pretrained_used = False
        if model_name == "resnet18":
            model = models.resnet18(weights=None)
            model.fc = nn.Linear(model.fc.in_features, 2)
        elif model_name == "mobilenet_v2":
            model = models.mobilenet_v2(weights=None)
            model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
        elif model_name == "efficientnet_b0":
            model = models.efficientnet_b0(weights=None)
            model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
        else:
            raise
    return model, pretrained_used


def set_backbone_trainable(model, model_name: str, trainable: bool):
    for param in model.parameters():
        param.requires_grad = trainable
    if model_name == "resnet18":
        for param in model.fc.parameters():
            param.requires_grad = True
    else:
        for param in model.classifier.parameters():
            param.requires_grad = True


def class_weights(rows):
    labels = [LABEL_TO_INDEX[row["label"]] for row in rows]
    counts = np.bincount(labels, minlength=2)
    weights = counts.sum() / np.maximum(counts, 1)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)


def run_epoch(model, dataloader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, labels)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += int((logits.argmax(dim=1) == labels).sum().item())
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate_split(model, dataloader):
    model.eval()
    y_true = []
    y_pred = []
    for images, labels in dataloader:
        images = images.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy().tolist()
        y_pred.extend(predictions)
        y_true.extend(labels.numpy().tolist())
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    acc = float(np.mean(y_true_arr == y_pred_arr))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_arr, y_pred_arr, labels=[0, 1], average="binary", pos_label=1, zero_division=0
    )
    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=[0, 1])
    return {
        "accuracy": acc,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm,
    }


def plot_training_curve(history, output_path: Path, title: str):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, [row["train_acc"] * 100 for row in history], label="Train")
    axes[0].plot(epochs, [row["val_acc"] * 100 for row in history], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].legend()

    axes[1].plot(epochs, [row["train_loss"] for row in history], label="Train")
    axes[1].plot(epochs, [row["val_loss"] for row in history], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close(fig)


def plot_confusion_matrix(cm: np.ndarray, output_path: Path, title: str):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(4.5, 4))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.xticks([0, 1], [INDEX_TO_LABEL[0], INDEX_TO_LABEL[1]])
    plt.yticks([0, 1], [INDEX_TO_LABEL[0], INDEX_TO_LABEL[1]])
    for y in range(cm.shape[0]):
        for x in range(cm.shape[1]):
            plt.text(x, y, str(cm[y, x]), ha="center", va="center")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()

Training device: cuda


In [24]:
# 11. Baseline training function
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def train_one_model(
    model_name: str,
    image_size: int = DEFAULT_IMAGE_SIZE,
    batch_size: int = DEFAULT_BATCH_SIZE,
    epochs: int = DEFAULT_EPOCHS,
    freeze_epochs: int = DEFAULT_FREEZE_EPOCHS,
    patience: int = DEFAULT_PATIENCE,
    lr: float = DEFAULT_LR,
    num_workers: int = NUM_WORKERS,
):
    set_seed(SEED)
    effective_batch_size = batch_size
    loaders = make_loaders(effective_batch_size, image_size=image_size, num_workers=num_workers)

    try:
        model, pretrained_used = build_model(model_name, pretrained=True)
        model = model.to(device)
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower() and effective_batch_size > 16:
            effective_batch_size = 16
            torch.cuda.empty_cache()
            loaders = make_loaders(effective_batch_size, image_size=image_size, num_workers=num_workers)
            model, pretrained_used = build_model(model_name, pretrained=True)
            model = model.to(device)
        else:
            raise

    criterion = nn.CrossEntropyLoss(weight=class_weights(ROWS_BY_SPLIT["train"]))
    set_backbone_trainable(model, model_name, trainable=False)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=1, factor=0.5)

    best_state = deepcopy(model.state_dict())
    best_epoch = 0
    best_train_acc = -1.0
    best_val_acc = -1.0
    epochs_without_improvement = 0
    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        if epoch == freeze_epochs + 1:
            set_backbone_trainable(model, model_name, trainable=True)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=1, factor=0.5)

        train_loss, train_acc = run_epoch(model, loaders["train"], criterion, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, loaders["val"], criterion, optimizer=None)
        scheduler.step(val_acc)

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
        })
        print(
            f"{model_name} epoch {epoch:02d}: "
            f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f}"
        )

        if val_acc > best_val_acc:
            best_epoch = epoch
            best_train_acc = float(train_acc)
            best_val_acc = float(val_acc)
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered for {model_name} at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    test_metrics = evaluate_split(model, loaders["test"])

    curve_path = LOCAL_FIGURES_DIR / f"{model_name}_training_curve.png"
    cm_path = LOCAL_FIGURES_DIR / f"{model_name}_test_confusion_matrix.png"
    plot_training_curve(history, curve_path, f"{model_name} Training Curve")
    plot_confusion_matrix(test_metrics["confusion_matrix"], cm_path, f"{model_name} Test Confusion Matrix")

    checkpoint_path = LOCAL_CHECKPOINT_DIR / f"{model_name}_best.pt"
    torch.save({
        "model": model_name,
        "model_state_dict": model.state_dict(),
        "history": history,
        "best_epoch": best_epoch,
        "class_to_index": LABEL_TO_INDEX,
        "image_size": image_size,
        "batch_size": effective_batch_size,
        "pretrained_used": pretrained_used,
    }, checkpoint_path)

    metrics = {
        "model": model_name,
        "best_epoch": best_epoch,
        "batch_size": effective_batch_size,
        "image_size": image_size,
        "pretrained_used": pretrained_used,
        "duration_seconds": round(time.time() - start_time, 2),
        "train_accuracy": best_train_acc,
        "val_accuracy": best_val_acc,
        "test_accuracy": float(test_metrics["accuracy"]),
        "test_precision": float(test_metrics["precision"]),
        "test_recall": float(test_metrics["recall"]),
        "test_f1": float(test_metrics["f1"]),
        "test_confusion_matrix": test_metrics["confusion_matrix"].tolist(),
    }

    with (LOCAL_RESULTS_DIR / f"{model_name}_history.json").open("w") as f:
        json.dump(history, f, indent=2)
    with (LOCAL_RESULTS_DIR / f"{model_name}_metrics.json").open("w") as f:
        json.dump(metrics, f, indent=2)

    return metrics

In [25]:
# 12. Run the three Stage 7 baselines
metrics_by_model = {}
for display_name, model_name in MODEL_ROWS:
    print("=" * 80)
    print(f"Training {display_name} / {model_name}")
    metrics_by_model[model_name] = train_one_model(
        model_name=model_name,
        image_size=DEFAULT_IMAGE_SIZE,
        batch_size=DEFAULT_BATCH_SIZE,
        epochs=DEFAULT_EPOCHS,
        freeze_epochs=DEFAULT_FREEZE_EPOCHS,
        patience=DEFAULT_PATIENCE,
        lr=DEFAULT_LR,
        num_workers=NUM_WORKERS,
    )

Training CNN-1 (ResNet18) / resnet18
resnet18 epoch 01: train_acc=0.9103 val_acc=0.9406 train_loss=0.3311 val_loss=0.2437
resnet18 epoch 02: train_acc=0.9810 val_acc=0.9882 train_loss=0.0625 val_loss=0.0416
resnet18 epoch 03: train_acc=0.9874 val_acc=0.9850 train_loss=0.0379 val_loss=0.0726
resnet18 epoch 04: train_acc=0.9892 val_acc=0.9885 train_loss=0.0317 val_loss=0.0592
resnet18 epoch 05: train_acc=0.9904 val_acc=0.9883 train_loss=0.0301 val_loss=0.0419
resnet18 epoch 06: train_acc=0.9906 val_acc=0.9880 train_loss=0.0258 val_loss=0.0582
Early stopping triggered for resnet18 at epoch 6.
Training CNN-2 (MobileNetV2) / mobilenet_v2
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 86.9MB/s]


mobilenet_v2 epoch 01: train_acc=0.9205 val_acc=0.9004 train_loss=0.3774 val_loss=0.3503
mobilenet_v2 epoch 02: train_acc=0.9822 val_acc=0.9780 train_loss=0.0636 val_loss=0.0739
mobilenet_v2 epoch 03: train_acc=0.9870 val_acc=0.9825 train_loss=0.0386 val_loss=0.0637
mobilenet_v2 epoch 04: train_acc=0.9897 val_acc=0.9848 train_loss=0.0291 val_loss=0.0584
mobilenet_v2 epoch 05: train_acc=0.9914 val_acc=0.9765 train_loss=0.0226 val_loss=0.0708
mobilenet_v2 epoch 06: train_acc=0.9914 val_acc=0.9746 train_loss=0.0236 val_loss=0.0662
Early stopping triggered for mobilenet_v2 at epoch 6.
Training CNN-3 (EfficientNet-B0) / efficientnet_b0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 234MB/s]


efficientnet_b0 epoch 01: train_acc=0.9081 val_acc=0.9309 train_loss=0.3414 val_loss=0.2317
efficientnet_b0 epoch 02: train_acc=0.9814 val_acc=0.9287 train_loss=0.0595 val_loss=0.1456
efficientnet_b0 epoch 03: train_acc=0.9876 val_acc=0.9908 train_loss=0.0367 val_loss=0.0378
efficientnet_b0 epoch 04: train_acc=0.9894 val_acc=0.9692 train_loss=0.0282 val_loss=0.0732
efficientnet_b0 epoch 05: train_acc=0.9910 val_acc=0.9809 train_loss=0.0225 val_loss=0.0649
Early stopping triggered for efficientnet_b0 at epoch 5.


In [26]:
# 13. Save Stage 7 results files locally
import pandas as pd

initial_results_rows = []
for display_name, model_name in MODEL_ROWS:
    metrics = metrics_by_model[model_name]
    initial_results_rows.append({
        "cnn_architecture": display_name,
        "model": model_name,
        "train_accuracy": metrics["train_accuracy"],
        "validation_accuracy": metrics["val_accuracy"],
        "test_accuracy": metrics["test_accuracy"],
        "precision": metrics["test_precision"],
        "recall": metrics["test_recall"],
        "f1": metrics["test_f1"],
    })

results_df = pd.DataFrame(initial_results_rows)
results_csv_path = LOCAL_RESULTS_DIR / "initial_results.csv"
metrics_summary_path = LOCAL_RESULTS_DIR / "metrics_summary.json"
results_df.to_csv(results_csv_path, index=False)
with metrics_summary_path.open("w") as f:
    json.dump(metrics_by_model, f, indent=2)

print(results_df)
print(f"Wrote {results_csv_path}")
print(f"Wrote {metrics_summary_path}")

          cnn_architecture            model  train_accuracy  \
0         CNN-1 (ResNet18)         resnet18        0.989175   
1      CNN-2 (MobileNetV2)     mobilenet_v2        0.989673   
2  CNN-3 (EfficientNet-B0)  efficientnet_b0        0.987635   

   validation_accuracy  test_accuracy  precision    recall        f1  
0             0.988529       0.993724   0.964744  0.978862  0.971751  
1             0.984818       0.987538   0.917368  0.974797  0.945211  
2             0.990778       0.992021   0.948154  0.981301  0.964443  
Wrote /content/stage7_outputs/artifacts/results/initial_results.csv
Wrote /content/stage7_outputs/artifacts/results/metrics_summary.json


In [27]:
# 14. Build the required markdown summary file

def pct(value: float) -> str:
    return f"{value * 100:.2f}%"

best_display_name, best_model_name = max(
    MODEL_ROWS,
    key=lambda item: metrics_by_model[item[1]]["test_accuracy"],
)
best_metrics = metrics_by_model[best_model_name]
train_val_gap = best_metrics["train_accuracy"] - best_metrics["val_accuracy"]

summary_lines = [
    "# Initial Experiment Summary",
    "",
    "## A. Task Type",
    "",
    "**Image Classification**",
    "",
    "## B. Experimental settings",
    "",
    "The initial Stage 7 experiment uses the reconstructed YawDD+ Dash mouth-crop dataset uploaded to Google Drive and trains directly from a copied local Colab workspace under `/content/` for faster I/O. Labels are read from the existing Stage 6 subject-level split manifest and are defined as `no_yawn` and `yawn`. The mouth crops come from the completed Stage 5 preprocessing pipeline, where the mouth ROI was extracted from MediaPipe Face Mesh landmarks with fallback lower-face crops already handled before Stage 7. Training uses the existing leakage-safe subject-level split, the three CNN baselines CNN-1 ResNet18, CNN-2 MobileNetV2, and CNN-3 EfficientNet-B0, Adam optimizer, learning rate 1e-4, batch size 32 with fallback to 16 if needed, weighted cross-entropy loss, up to 12 epochs, early stopping patience 3, ReduceLROnPlateau scheduling, and realistic training-only augmentation with small rotation, mild brightness/contrast jitter, and slight affine scaling.",
    "",
    "## C. Initial Results table",
    "",
    "| CNN Architecture | Train Accuracy | Validation Accuracy | Test Accuracy |",
    "|---|---:|---:|---:|",
]

for display_name, model_name in MODEL_ROWS:
    row = metrics_by_model[model_name]
    summary_lines.append(
        f"| {display_name} | {pct(row['train_accuracy'])} | {pct(row['val_accuracy'])} | {pct(row['test_accuracy'])} |"
    )

summary_lines.extend([
    "",
    "## D. Short interpretation",
    "",
    f"{best_display_name} achieved the strongest test accuracy in this initial Stage 7 run.",
    f"The best model shows a train-validation gap of {train_val_gap * 100:.2f} percentage points, which is the main early signal to monitor for overfitting.",
    f"Class imbalance still matters because `no_yawn` is more frequent than `yawn`, so the weighted cross-entropy setting and the test confusion matrix should be reviewed together rather than relying on accuracy alone.",
    "Because the split is done at subject level rather than image level, the evaluation is more challenging and more realistic than a random frame split.",
    "The next step should be focused error analysis on the saved confusion matrices and misclassified subjects, followed by controlled hyperparameter tuning or additional data balancing experiments.",
])

summary_path = LOCAL_REPORTS_DIR / "initial_experiment_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n")
print(summary_path.read_text())

# Initial Experiment Summary

## A. Task Type

**Image Classification**

## B. Experimental settings

The initial Stage 7 experiment uses the reconstructed YawDD+ Dash mouth-crop dataset uploaded to Google Drive and trains directly from a copied local Colab workspace under `/content/` for faster I/O. Labels are read from the existing Stage 6 subject-level split manifest and are defined as `no_yawn` and `yawn`. The mouth crops come from the completed Stage 5 preprocessing pipeline, where the mouth ROI was extracted from MediaPipe Face Mesh landmarks with fallback lower-face crops already handled before Stage 7. Training uses the existing leakage-safe subject-level split, the three CNN baselines CNN-1 ResNet18, CNN-2 MobileNetV2, and CNN-3 EfficientNet-B0, Adam optimizer, learning rate 1e-4, batch size 32 with fallback to 16 if needed, weighted cross-entropy loss, up to 12 epochs, early stopping patience 3, ReduceLROnPlateau scheduling, and realistic training-only augmentation with small

In [28]:
# 15. Copy outputs back to Google Drive
import shutil

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

for src in LOCAL_RESULTS_DIR.glob("*"):
    shutil.copy2(src, DRIVE_RESULTS_DIR / src.name)

for src in LOCAL_FIGURES_DIR.glob("*"):
    shutil.copy2(src, DRIVE_FIGURES_DIR / src.name)

for src in LOCAL_CHECKPOINT_DIR.glob("*"):
    shutil.copy2(src, DRIVE_CHECKPOINT_DIR / src.name)

for src in LOCAL_REPORTS_DIR.glob("*"):
    shutil.copy2(src, DRIVE_REPORTS_DIR / src.name)

print("Saved Stage 7 outputs back to Drive:")
print("-", DRIVE_RESULTS_DIR)
print("-", DRIVE_FIGURES_DIR)
print("-", DRIVE_CHECKPOINT_DIR)
print("-", DRIVE_REPORTS_DIR)

Saved Stage 7 outputs back to Drive:
- /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/results
- /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/figures
- /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/checkpoints
- /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/reports


In [29]:
# 16. Final summary section
from IPython.display import display

final_results_path = DRIVE_RESULTS_DIR / "initial_results.csv"
summary_report_path = DRIVE_REPORTS_DIR / "initial_experiment_summary.md"

final_df = pd.read_csv(final_results_path)
display(final_df)

stage7_success = final_results_path.exists() and summary_report_path.exists()
print("\nStage 7 completion status:", "SUCCESS" if stage7_success else "INCOMPLETE")
print("Results CSV:", final_results_path)
print("Metrics JSON:", DRIVE_RESULTS_DIR / "metrics_summary.json")
print("Figures dir:", DRIVE_FIGURES_DIR)
print("Checkpoints dir:", DRIVE_CHECKPOINT_DIR)
print("Summary report:", summary_report_path)

,cnn_architecture,model,train_accuracy,validation_accuracy,test_accuracy,precision,recall,f1
0,CNN-1 (ResNet18),resnet18,0.989175,0.988529,0.993724,0.964744,0.978862,0.971751
1,CNN-2 (MobileNetV2),mobilenet_v2,0.989673,0.984818,0.987538,0.917368,0.974797,0.945211
2,CNN-3 (EfficientNet-B0),efficientnet_b0,0.987635,0.990778,0.992021,0.948154,0.981301,0.964443



Stage 7 completion status: SUCCESS
Results CSV: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/results/initial_results.csv
Metrics JSON: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/results/metrics_summary.json
Figures dir: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/figures
Checkpoints dir: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/checkpoints
Summary report: /content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/reports/initial_experiment_summary.md
